# ============================================
# 03_open_power_system_data_time_series_downloader.ipynb
# Open Power System Data - Time Series
# ============================================

## Dataset Title
Open Power System Data - Time Series

## Source URL
Official dataset page:  
https://data.open-power-system-data.org/time_series/2018-03-13

Direct package ZIP:  
https://data.open-power-system-data.org/time_series/2018-03-13/opsd-time_series-2018-03-13.zip

Recommended direct CSV (60-minute singleindex):  
https://data.open-power-system-data.org/time_series/2018-03-13/time_series_60min_singleindex.csv

## Purpose of This Dataset
This dataset is used for real power-system time-series analysis in the PhD workflow.

It is especially useful for:
- renewable energy forecasting
- load-demand modeling
- solar and wind generation analysis
- system-level energy representation
- baseline forecasting and optimization preparation

## Why We Need This Dataset
We need this dataset because it contains real, structured time-series data relevant to renewable-aware energy systems.

According to the Open Power System Data platform, this package includes:
- electricity consumption (load)
- wind generation
- solar generation
- prices
- multiple European countries
- hourly and higher-resolution variants

This makes it highly suitable for:
- Notebook 3: NEO-Forecast
- Notebook 4: System Modeling
- Notebook 6: NEO-Opt
- Notebook 7: NEO-Robust
- Notebook 11: Full System Simulation

## What We Are Doing in This Notebook
In this notebook, we will:

1. Mount Google Drive
2. Create dataset folders
3. Download the real dataset from the official source
4. Save both:
   - the package ZIP
   - the recommended 60-minute CSV
5. Extract the ZIP package
6. Save a dataset manifest
7. Build a clean ZIP package of usable files
8. Save everything in Google Drive
9. Download the final package to the local PC

## Where This Data Will Be Used Next
This dataset will be used in:

- Notebook 3: NEO-Forecast
- Notebook 4: System Modeling
- Notebook 6: NEO-Opt
- Notebook 7: NEO-Robust
- Notebook 11: Full System Simulation

## Notes
The OPSD time-series package page states that the package contains load, wind and solar, and prices, and offers multiple downloadable formats including the 60-minute singleindex CSV and the package ZIP. The page also lists the 60-minute singleindex CSV as 121 MB. :contentReference[oaicite:1]{index=1}

In [1]:
# ============================================
# 03_open_power_system_data_time_series_downloader.ipynb
# FIXED CSV-ONLY SINGLE CELL
# ============================================

import os
import json
import shutil
import zipfile
from pathlib import Path
import requests
from google.colab import drive, files

# -----------------------------
# 1. Dataset metadata
# -----------------------------
DATASET_NAME = "Open Power System Data - Time Series"
SOURCE_PAGE = "https://data.open-power-system-data.org/time_series/"
CSV_60MIN_URL = "https://data.open-power-system-data.org/time_series/2020-10-06/time_series_60min_singleindex.csv"

print("Dataset Name :", DATASET_NAME)
print("Source Page  :", SOURCE_PAGE)
print("CSV URL      :", CSV_60MIN_URL)

# -----------------------------
# 2. Mount Google Drive
# -----------------------------
drive.mount('/content/drive')

# -----------------------------
# 3. Create folder structure
# -----------------------------
project_root = Path("/content/drive/MyDrive/PhD_Thesis_Datasets")
dataset_root = project_root / "03_open_power_system_data_time_series"
raw_dir = dataset_root / "raw"
bundle_dir = dataset_root / "bundle"

for folder in [project_root, dataset_root, raw_dir, bundle_dir]:
    folder.mkdir(parents=True, exist_ok=True)

print("\nCreated folders:")
print(" -", project_root)
print(" -", dataset_root)
print(" -", raw_dir)
print(" -", bundle_dir)

# -----------------------------
# 4. Robust download helper
# -----------------------------
def download_file(url: str, dest_path: Path, chunk_size: int = 1024 * 1024):
    headers = {"User-Agent": "Mozilla/5.0"}
    with requests.get(url, headers=headers, stream=True, timeout=600) as r:
        r.raise_for_status()
        with open(dest_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)
    return dest_path

# -----------------------------
# 5. Download CSV
# -----------------------------
csv_path = raw_dir / "time_series_60min_singleindex.csv"

if not csv_path.exists():
    print("\nDownloading OPSD 60-minute CSV...")
    download_file(CSV_60MIN_URL, csv_path)
    print("CSV downloaded:", csv_path)
else:
    print("\nUsing cached CSV:", csv_path)

if not csv_path.exists():
    raise FileNotFoundError(f"CSV not found: {csv_path}")

print("CSV size:", round(csv_path.stat().st_size / (1024*1024), 2), "MB")

# -----------------------------
# 6. Save manifest
# -----------------------------
manifest = {
    "dataset_name": DATASET_NAME,
    "source_page": SOURCE_PAGE,
    "csv_60min_url": CSV_60MIN_URL,
    "downloaded_csv_60min": str(csv_path),
    "used_in_next_notebooks": [
        "Notebook 3: NEO-Forecast",
        "Notebook 4: System Modeling",
        "Notebook 6: NEO-Opt",
        "Notebook 7: NEO-Robust",
        "Notebook 11: Full System Simulation"
    ]
}

manifest_path = dataset_root / "dataset_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("Saved manifest:", manifest_path)

# -----------------------------
# 7. Create clean bundle ZIP in Google Drive
# -----------------------------
clean_bundle_zip_path = bundle_dir / "opsd_time_series_CLEAN_bundle.zip"

with zipfile.ZipFile(clean_bundle_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(csv_path, arcname="raw/" + csv_path.name)
    zf.write(manifest_path, arcname="manifest/" + manifest_path.name)

print("Created clean bundle ZIP:", clean_bundle_zip_path)

# -----------------------------
# 8. Create full package ZIP in Google Drive
# -----------------------------
full_zip_path = dataset_root / "opsd_time_series_FULL_PACKAGE.zip"

with zipfile.ZipFile(full_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(csv_path, arcname="raw/" + csv_path.name)
    zf.write(clean_bundle_zip_path, arcname="bundle/" + clean_bundle_zip_path.name)
    zf.write(manifest_path, arcname="manifest/" + manifest_path.name)

print("Created full package ZIP in Google Drive:", full_zip_path)
print("Full package size:", round(full_zip_path.stat().st_size / (1024*1024), 2), "MB")

# -----------------------------
# 9. Copy ZIP to /content for reliable PC download
# -----------------------------
runtime_copy = Path("/content/opsd_time_series_FULL_PACKAGE.zip")
shutil.copy2(full_zip_path, runtime_copy)

print("Runtime backup created:", runtime_copy)
print("Runtime backup size:", round(runtime_copy.stat().st_size / (1024*1024), 2), "MB")

# -----------------------------
# 10. Final summary
# -----------------------------
print("\n===== SUMMARY =====")
print("Dataset root         :", dataset_root)
print("Raw CSV              :", csv_path)
print("Manifest             :", manifest_path)
print("Clean bundle ZIP     :", clean_bundle_zip_path)
print("Full package ZIP     :", full_zip_path)
print("Runtime backup ZIP   :", runtime_copy)
print("===================")

# -----------------------------
# 11. Download final package to PC
# -----------------------------
print("\nStarting browser download...")
files.download(str(runtime_copy))

Dataset Name : Open Power System Data - Time Series
Source Page  : https://data.open-power-system-data.org/time_series/
CSV URL      : https://data.open-power-system-data.org/time_series/2020-10-06/time_series_60min_singleindex.csv
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Created folders:
 - /content/drive/MyDrive/PhD_Thesis_Datasets
 - /content/drive/MyDrive/PhD_Thesis_Datasets/03_open_power_system_data_time_series
 - /content/drive/MyDrive/PhD_Thesis_Datasets/03_open_power_system_data_time_series/raw
 - /content/drive/MyDrive/PhD_Thesis_Datasets/03_open_power_system_data_time_series/bundle

CSV downloaded: /content/drive/MyDrive/PhD_Thesis_Datasets/03_open_power_system_data_time_series/raw/time_series_60min_singleindex.csv
CSV size: 124.3 MB
Saved manifest: /content/drive/MyDrive/PhD_Thesis_Datasets/03_open_power_system_data_time_series/dataset_manifest.json
Created clean bundle ZIP: /content/dri

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>